# 01 – Preprocessing
**Goal:** turn the 84 selected ICU stays into clean, comparable tables with one row per 15 minutes, ready for gap filling (notebook 02) and the anomaly models (notebooks 03–05).

**Why:** the raw numerics files are messy. The rows are shuffled in time, ~85% of the cuff values are copies, sensors sometimes produce impossible values, and every signal is measured at a different rhythm (every second vs. every ~20 minutes). The models need one clean row per fixed time step.

**Input:** the 84 stays selected in notebook 00 (`selected_stays.csv`, decision D4).
**Output:** one table per stay on a 15-minute grid, with a marker for which values were really measured, labelled events, and a train/test split per patient.

**Steps:**
1. Setup and the selected stays
2. Loading one stay (sorted by time, hours since start)
3. Removing the cuff copies
4. Removing impossible values
5. Resampling to a 15-minute grid (D7), keeping track of what was really measured
6. Labelling deterioration events (D8)
7. Train/test split per patient (D6)
8. Scaling on the training patients only
9. Saving the processed data

The reasoning behind each choice is in `decisions.md`.

## 1. Setup and the selected stays

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = os.path.join("..", "dataset", "mimic4wdb")
TICKS_PER_SECOND = 999.56
SIGNALS = ["HR [bpm]", "RR [rpm]", "SpO2 [%]", "NBPs [mmHg]", "NBPd [mmHg]"]
NBP_COLS = ["NBPs [mmHg]", "NBPd [mmHg]", "NBPm [mmHg]"]

selected = pd.read_csv(os.path.join("..", "selected_stays.csv"))
print(len(selected), "selected stays")
selected.head()

84 selected stays


,file,patient
0,p10014354_81739927n.csv.gz,p10014354
1,p10019003_87033314n.csv.gz,p10019003
2,p10039708_83411188n.csv.gz,p10039708
3,p10079700_85594648n.csv.gz,p10079700
4,p10112163_88501826n.csv.gz,p10112163


## 2. Loading one stay
`load_stay` reads one stay with only the columns we need, sorts the rows by time (the files are shuffled) and adds `hours` since the start of the stay (see "Data conventions" in `decisions.md`).

In [11]:
def load_stay(file):
    """Load one stay: our columns only, sorted by time, with hours since start."""
    d = pd.read_csv(os.path.join(DATA_DIR, file),
                    usecols=lambda c: c == "time" or c in SIGNALS or c in NBP_COLS)
    d = d.sort_values("time").reset_index(drop=True)
    d["hours"] = (d["time"] - d["time"].min()) / TICKS_PER_SECOND / 3600
    return d

example = load_stay(selected["file"].iloc[0])
print(example.shape)
print("rows sorted by time:", example["time"].is_monotonic_increasing)
example.head()

(258061, 8)
rows sorted by time: True


,time,HR [bpm],NBPd [mmHg],NBPm [mmHg],NBPs [mmHg],RR [rpm],SpO2 [%],hours
0,33600,NaN,NaN,NaN,NaN,NaN,95.1,0.000000
1,34624,NaN,NaN,NaN,NaN,NaN,95.1,0.000285
2,35648,NaN,NaN,NaN,NaN,NaN,95.1,0.000569
3,36672,NaN,NaN,NaN,NaN,NaN,95.1,0.000854
4,37696,NaN,NaN,NaN,NaN,NaN,95.1,0.001138


## 3. Removing the cuff copies
The monitor re-writes the last cuff value every ~2 minutes (notebook 00, section 9). We keep only real cuff measurements: a cuff value that is identical to the previous cuff row is a copy. We make only the cuff cells of a copy empty (`NaN`), not the whole row, because HR, RR and SpO2 in that row are real.

In [12]:
def remove_cuff_copies(d):
    """Make cuff values empty (NaN) where they repeat the previous cuff row."""
    d = d.copy()
    cols = [c for c in NBP_COLS if c in d.columns]
    cuff = d.loc[d["NBPs [mmHg]"].notna(), cols].fillna(-1)
    is_copy = ~cuff.ne(cuff.shift()).any(axis=1)
    d.loc[cuff.index[is_copy], cols] = np.nan
    return d

before = example["NBPs [mmHg]"].notna().sum()
example = remove_cuff_copies(example)
after = example["NBPs [mmHg]"].notna().sum()
print(f"cuff values: {before} before, {after} after ({100 * (1 - after / before):.0f}% were copies)")

cuff values: 726 before, 149 after (79% were copies)


**Check (first stay):** 726 cuff values before, 149 after → 79% were copies, in line with notebook 00 (section 9).

## 4. Removing impossible values
Sensors sometimes produce impossible values (e.g. a loose finger clip gives SpO2 = 0). Left in, the models would see a sensor error as deterioration. We use wide plausibility ranges, so only clearly impossible values are removed and real extreme values (which we want to detect) stay. As with the copies, we only empty the cell, not the whole row. The ranges are a choice, documented in `decisions.md`.

In [13]:
VALID_RANGES = {
    "HR [bpm]": (20, 250),
    "RR [rpm]": (3, 60),
    "SpO2 [%]": (50, 100),
    "NBPs [mmHg]": (40, 260),
    "NBPd [mmHg]": (20, 180),
}

def remove_impossible(d):
    """Make values outside the plausible range empty (NaN). Returns the table and the counts."""
    d = d.copy()
    removed = {}
    for col, (low, high) in VALID_RANGES.items():
        bad = d[col].notna() & ((d[col] < low) | (d[col] > high))
        removed[col] = int(bad.sum())
        d.loc[bad, col] = np.nan
    bad_bp = d["NBPs [mmHg]"].notna() & d["NBPd [mmHg]"].notna() & (d["NBPs [mmHg]"] <= d["NBPd [mmHg]"])
    removed["NBPs <= NBPd"] = int(bad_bp.sum())
    d.loc[bad_bp, ["NBPs [mmHg]", "NBPd [mmHg]"]] = np.nan
    return d, removed

example, removed = remove_impossible(example)
print("values removed:", removed)

values removed: {'HR [bpm]': 62, 'RR [rpm]': 26, 'SpO2 [%]': 0, 'NBPs [mmHg]': 0, 'NBPd [mmHg]': 0, 'NBPs <= NBPd': 0}


In [14]:
raw = load_stay(selected["file"].iloc[0])
print(raw.loc[(raw["HR [bpm]"] < 20) | (raw["HR [bpm]"] > 250), "HR [bpm]"].value_counts().head())

HR [bpm]
0.0    62
Name: count, dtype: int64


**Check (first stay):** 62 HR and 26 RR values removed (< 0.03% of the rows); no impossible SpO2 or cuff values.

## 5. Resampling to a 15-minute grid (D7)
The models compare the five signals at the same moment (e.g. HR rising while blood pressure falls), so all signals share one grid of 15-minute blocks, with one row per block.

Per block and per signal we store:
- the **median**: the typical value (robust to outliers, unlike the mean);
- the **minimum** and **maximum**: so short dips or peaks stay visible (e.g. 2 minutes of SpO2 < 90% would disappear in an average);
- the **standard deviation (std)**: how much the signal jumped around within the block;
- **measured** (True/False): whether the block contains a real measurement.

HR, RR and SpO2 have ~900 values per block (one per second); the cuff usually 0, 1 or 2. Blocks without a measurement stay empty (`NaN`); they are filled in notebook 02.

In [15]:
GRID_MIN = 15

def to_grid(d):
    """Per 15-minute block: median, min, max and std of each signal; empty blocks stay NaN."""
    block = (d["hours"] * 60 // GRID_MIN).astype(int)
    groups = d.groupby(block)[SIGNALS]
    grid = groups.median()
    grid = grid.join(groups.min().add_suffix(" min"))
    grid = grid.join(groups.max().add_suffix(" max"))
    grid = grid.join(groups.std().add_suffix(" std"))
    grid = grid.reindex(range(block.max() + 1))
    grid.index.name = "block"
    for s in SIGNALS:
        grid[s + " measured"] = grid[s].notna()
    grid["hours"] = grid.index * GRID_MIN / 60
    return grid

grid = to_grid(example)
print(grid.shape)
print((100 * grid[[s + " measured" for s in SIGNALS]].mean()).round(1))
grid[["HR [bpm]", "HR [bpm] min", "HR [bpm] max", "HR [bpm] std", "HR [bpm] measured"]].head()

(119, 26)
HR [bpm] measured       100.0
RR [rpm] measured       100.0
SpO2 [%] measured       100.0
NBPs [mmHg] measured     63.9
NBPd [mmHg] measured     60.5
dtype: float64


,HR [bpm],HR [bpm] min,HR [bpm] max,HR [bpm] std,HR [bpm] measured
block,,,,,
0,128.0,112.0,141.0,7.497998,True
1,121.0,112.0,138.0,3.273990,True
2,120.0,113.0,130.0,3.344773,True
3,113.0,105.0,136.0,5.895821,True
4,113.0,105.0,136.0,6.414465,True


**Check (first stay):** 119 blocks (~30 h), 26 columns. HR, RR and SpO2 are measured in 100% of the blocks, the cuff in ~60–64% (this patient's cuff measured about every 12 minutes). Some blocks have a systolic but no diastolic value. For the cuff, the std is usually empty (only one measurement per block).
Note for D8: this patient starts with HR ≈ 129 bpm, above the event threshold, so the first hours of a stay are not always a stable baseline.

### 5.1 All 84 stays
We apply steps 2–5 to every selected stay and combine the results in one table (`grids`), with one row per 15-minute block per stay.

In [18]:
all_grids = []
removed_total = {}

for f in selected["file"]:
    d = load_stay(f)
    d = remove_cuff_copies(d)
    d, removed = remove_impossible(d)
    for k, v in removed.items():
        removed_total[k] = removed_total.get(k, 0) + v
    g = to_grid(d)
    g["file"] = f
    g["patient"] = f.split("_")[0]
    all_grids.append(g)

grids = pd.concat(all_grids).reset_index()
print(len(selected), "stays,", len(grids), "blocks of 15 minutes in total")
print("impossible values removed:", removed_total)
print((100 * grids[[s + " measured" for s in SIGNALS]].mean()).round(1))

84 stays, 16973 blocks of 15 minutes in total
impossible values removed: {'HR [bpm]': 2585, 'RR [rpm]': 28356, 'SpO2 [%]': 376, 'NBPs [mmHg]': 2, 'NBPd [mmHg]': 7, 'NBPs <= NBPd': 0}
HR [bpm] measured       99.6
RR [rpm] measured       98.4
SpO2 [%] measured       98.8
NBPs [mmHg] measured    47.3
NBPd [mmHg] measured    46.9
dtype: float64


**Result:** 84 stays, 16,973 blocks of 15 minutes (~4,240 hours, ~50 h per stay on average).
- HR, RR and SpO2 are measured in 98–100% of the blocks; the cuff in ~47%.
- Impossible values removed: HR 2,585, RR 28,356, SpO2 376, cuff 9. RR has by far the most (still ~0.2% of all RR values); see the check below.

In [19]:
rr_removed = []
for f in selected["file"]:
    d = remove_cuff_copies(load_stay(f))
    rr = d["RR [rpm]"]
    rr_removed.append(rr[(rr < 3) | (rr > 60)])
rr_removed = pd.concat(rr_removed)
print(len(rr_removed), "RR values removed")
print("RR = 0:", (rr_removed == 0).sum())
print("RR between 0 and 3:", ((rr_removed > 0) & (rr_removed < 3)).sum())
print("RR > 60:", (rr_removed > 60).sum())

28356 RR values removed
RR = 0: 24928
RR between 0 and 3: 2029
RR > 60: 1399


**RR check:** of the 28,356 removed RR values, 24,928 (88%) are exactly 0, 2,029 are between 0 and 3, and 1,399 are above 60. RR = 0 is most likely a sensor dropout (loose sticker or ventilated patient); a real apnea is possible but would also show in SpO2 and HR. We treat it as missing (D9).

### RR = 0: how long are the stretches?
We group consecutive RR = 0 values into stretches and measure how long each lasts. A real breathing pause (apnea) lasts seconds to about a minute; a stretch of many minutes is a sensor dropout.

In [20]:
lengths = []
for f in selected["file"]:
    d = load_stay(f)
    rr = d.loc[d["RR [rpm]"].notna(), ["hours", "RR [rpm]"]]
    zero = rr["RR [rpm]"] == 0
    run_id = (zero != zero.shift()).cumsum()
    for _, r in rr[zero].groupby(run_id[zero]):
        lengths.append((r["hours"].max() - r["hours"].min()) * 60)

lengths = pd.Series(lengths, name="minutes")
print(len(lengths), "stretches of RR = 0")
print("shorter than 1 minute:", (lengths < 1).sum())
print("1 to 5 minutes:", ((lengths >= 1) & (lengths <= 5)).sum())
print("longer than 5 minutes:", (lengths > 5).sum())
print("total time in stretches > 5 min:", round(lengths[lengths > 5].sum() / 60, 1), "hours")

1275 stretches of RR = 0
shorter than 1 minute: 1178
1 to 5 minutes: 86
longer than 5 minutes: 11
total time in stretches > 5 min: 2.7 hours


**Result:** 1,275 stretches of RR = 0. 11 last longer than 5 minutes (2.7 hours in total, ~40% of all zeros): certainly sensor dropouts. The large majority (1,178) last less than a minute; these could be real short breathing pauses or brief sensor glitches, so we check SpO2 next.

### RR = 0: does SpO2 drop?
If RR = 0 is a real breathing pause, oxygen saturation should fall shortly after. For each stretch we compare the median SpO2 in the 2 minutes before with the lowest SpO2 from its start until 2 minutes after its end. A drop of 4 points or more counts as a desaturation (a common clinical cutoff).

In [21]:
runs = []
for f in selected["file"]:
    d = load_stay(f)
    d.loc[(d["SpO2 [%]"] < 50) | (d["SpO2 [%]"] > 100), "SpO2 [%]"] = np.nan

    rr = d.loc[d["RR [rpm]"].notna(), ["hours", "RR [rpm]"]]
    zero = rr["RR [rpm]"] == 0
    run_id = (zero != zero.shift()).cumsum()

    s = d.loc[d["SpO2 [%]"].notna(), ["hours", "SpO2 [%]"]]
    sh, sv = s["hours"].values, s["SpO2 [%]"].values

    for _, r in rr[zero].groupby(run_id[zero]):
        start, end = r["hours"].min(), r["hours"].max()
        before = sv[np.searchsorted(sh, start - 2 / 60):np.searchsorted(sh, start)]
        during = sv[np.searchsorted(sh, start):np.searchsorted(sh, end + 2 / 60, side="right")]
        runs.append({
            "minutes": (end - start) * 60,
            "spo2_before": np.median(before) if len(before) else np.nan,
            "spo2_min_after": during.min() if len(during) else np.nan,
        })

runs = pd.DataFrame(runs)
runs["drop"] = runs["spo2_before"] - runs["spo2_min_after"]
runs["length"] = pd.cut(runs["minutes"], [-1, 1, 5, np.inf], labels=["< 1 min", "1–5 min", "> 5 min"])

runs.groupby("length", observed=True).agg(
    stretches=("minutes", "size"),
    median_drop=("drop", "median"),
    drop_4_or_more_pct=("drop", lambda x: round(100 * (x >= 4).mean(), 1)),
)

,stretches,median_drop,drop_4_or_more_pct
length,,,
< 1 min,1178,1.1,18.3
1–5 min,86,0.8,10.5
> 5 min,11,1.7,36.4


**Result:** short stretches of RR = 0 do not show more SpO2 drops than the long stretches we know are sensor dropouts (≥ 4 points: 18% for < 1 min vs. 36% for > 5 min; typical drop ~1 point). The higher share for long stretches is probably a window effect: a longer window more easily contains a low value, and it is based on only 11 stretches.

**Conclusion:** no evidence that RR = 0 stretches are real breathing pauses; they behave like sensor dropouts, so we treat RR = 0 as missing (D9). Limitations: we did not compare with random windows without RR = 0, and extra oxygen can keep SpO2 up during a short pause.